In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from typing import List, Optional, Dict, Tuple

def load_experiment_results(csv_path: str) -> pd.DataFrame:
    """
    Load experiment results from CSV file.
    
    Args:
        csv_path: Path to the experiment results CSV file
        
    Returns:
        DataFrame with experiment results
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Results file not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    # Calculate accuracy metrics if not already present
    if 'adaptive_incorrect_count' in df.columns and 'passed_judge_count' in df.columns:
        if 'adaptive_accuracy' not in df.columns:
            df['adaptive_accuracy'] = 1 - (df['adaptive_incorrect_count'] / df['passed_judge_count'])
        if 'transfer_accuracy' not in df.columns:
            df['transfer_accuracy'] = 1 - (df['re_eval_incorrect_count'] / df['passed_judge_count'])
        if 'transfer_error_rate' not in df.columns:
            df['transfer_error_rate'] = df['re_eval_incorrect_count'] / df['passed_judge_count']
    
    return df

def clean_model_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean model names for better display in plots.
    
    Args:
        df: DataFrame with model names
        
    Returns:
        DataFrame with cleaned model names
    """
    df_clean = df.copy()
    
    # Extract final part of model names and clean them
    for col in ['eval_model_name', 're_eval_model_name', 'generator_model_name']:
        if col in df_clean.columns:
            df_clean[col] = (
                df_clean[col]
                .str.split('/')
                .str[-1]
                .str.replace('Llama-3.3-70B-Instruct-Turbo', 'Llama-3.3 70B')
                .str.replace('claude-3-5-sonnet-latest', 'Claude 3.5 Sonnet')
                .str.replace('gpt-4o-mini', 'GPT-4o mini')
                .str.replace('gpt-4o', 'GPT-4o')
                .str.replace('Meta-Llama-3.1-405B-Instruct-Turbo', 'Llama-3.1 405B')
            )
    
    return df_clean


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from typing import List, Optional, Dict, Tuple

def load_experiment_results(csv_path: str) -> pd.DataFrame:
    """
    Load experiment results from CSV file.
    
    Args:
        csv_path: Path to the experiment results CSV file
        
    Returns:
        DataFrame with experiment results
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Results file not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    # Calculate accuracy metrics if not already present
    if 'adaptive_incorrect_count' in df.columns and 'passed_judge_count' in df.columns:
        if 'adaptive_accuracy' not in df.columns:
            df['adaptive_accuracy'] = 1 - (df['adaptive_incorrect_count'] / df['passed_judge_count'])
        if 'transfer_accuracy' not in df.columns:
            df['transfer_accuracy'] = 1 - (df['re_eval_incorrect_count'] / df['passed_judge_count'])
        if 'transfer_error_rate' not in df.columns:
            df['transfer_error_rate'] = df['re_eval_incorrect_count'] / df['passed_judge_count']
    
    return df

def clean_model_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean model names for better display in plots.
    
    Args:
        df: DataFrame with model names
        
    Returns:
        DataFrame with cleaned model names
    """
    df_clean = df.copy()
    
    # Extract final part of model names and clean them
    for col in ['eval_model_name', 're_eval_model_name', 'generator_model_name']:
        if col in df_clean.columns:
            df_clean[col] = (
                df_clean[col]
                .str.split('/')
                .str[-1]
                .str.replace('Llama-3.3-70B-Instruct-Turbo', 'Llama-3.3 70B')
                .str.replace('claude-3-5-sonnet-latest', 'Claude 3.5 Sonnet')
                .str.replace('gpt-4o-mini', 'GPT-4o mini')
                .str.replace('gpt-4o', 'GPT-4o')
                .str.replace('Meta-Llama-3.1-405B-Instruct-Turbo', 'Llama-3.1 405B')
            )
    
    return df_clean

def plot_transfer_matrix(
    df: pd.DataFrame,
    metric: str = 'transfer_error_rate',
    figsize: Tuple[int, int] = (10, 6),
    cmap: str = 'YlOrRd',
    vmin: float = 0,
    vmax: float = 1,
    task_name: Optional[str] = None,
    save_path: Optional[str] = None
) -> None:
    """
    Plot a transfer matrix showing how well models transfer to each other.
    
    Args:
        df: DataFrame with experiment results
        metric: Metric to plot ('transfer_error_rate' or 'transfer_accuracy')
        figsize: Figure size (width, height)
        cmap: Colormap for heatmap
        vmin: Minimum value for colormap
        vmax: Maximum value for colormap
        task_name: Optional task name for title
        save_path: Optional path to save the figure
    """
    df_plot = clean_model_names(df)
    
    # Define model orders (can be customized)
    eval_models = sorted(df_plot['eval_model_name'].unique())
    re_eval_models = sorted(df_plot['re_eval_model_name'].unique())
    
    # Pivot the data for heatmap format
    heatmap_data = df_plot.pivot(
        index='re_eval_model_name',
        columns='eval_model_name',
        values=metric
    )
    
    # Reorder the index and columns if needed
    if eval_models:
        heatmap_data = heatmap_data.reindex(columns=eval_models)
    if re_eval_models:
        heatmap_data = heatmap_data.reindex(index=re_eval_models)
    
    # Set style for paper-quality plots
    sns.set_style("white")
    plt.rcParams.update({
        'font.size': 18,
        'axes.labelsize': 18,
        'axes.titlesize': 24,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18
    })
    
    # Create figure
    plt.figure(figsize=figsize, dpi=300)
    
    # Create heatmap
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='.2f',
        cmap=cmap,
        cbar=False,
        square=False,
        vmin=vmin,
        vmax=vmax,
        annot_kws={'size': 22}
    )
    
    # Set labels
    metric_label = "Error Rate" if metric == 'transfer_error_rate' else "Accuracy"
    plt.xlabel('Target Model', fontsize=22, fontweight='bold')
    plt.ylabel('Evaluated Model', fontsize=22, fontweight='bold')
    
    if task_name:
        plt.title(f'Transfer {metric_label} - {task_name}', fontsize=24)
    
    # Keep ticks unrotated
    plt.xticks(rotation=15, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
    
    plt.show()

def plot_ablation_bars(
    df: pd.DataFrame,
    eval_model: str,
    metric: str = 'adaptive_accuracy',
    ablation_column: str = 'use_cot_target',
    figsize: Tuple[int, int] = (12, 7),
    task_name: Optional[str] = None,
    save_path: Optional[str] = None
) -> None:
    """
    Plot bar charts for ablation studies.
    
    Args:
        df: DataFrame with experiment results
        eval_model: Model to filter for (or 'all' for all models)
        metric: Metric to plot ('adaptive_accuracy' or 'transfer_accuracy')
        ablation_column: Column to use for ablation comparison
        figsize: Figure size (width, height)
        task_name: Optional task name for title
        save_path: Optional path to save the figure
    """
    df_plot = clean_model_names(df)
    
    # Filter for the specified evaluation model
    if eval_model != 'all':
        df_plot = df_plot[df_plot['eval_model_name'].str.contains(eval_model)]
    
    # Create a new column for the ablation condition
    ablation_label = {
        'use_cot_target': 'Target CoT',
        'use_cot_in_context_attacker': 'Generator CoT',
        'use_example': 'With Examples'
    }.get(ablation_column, ablation_column)
    
    df_plot[ablation_label] = df_plot[ablation_column].map({True: 'Yes', False: 'No'})
    
    # Set style
    sns.set_style("white")
    plt.figure(figsize=figsize, dpi=300)
    
    # Create the bar plot
    ax = sns.barplot(
        data=df_plot,
        x='generator_model_name',
        y=metric,
        hue=ablation_label,
        palette=['#1f77b4', '#ff7f0e']
    )
    
    # Adjust labels and title
    plt.xlabel('Generator Model', fontsize=16)
    
    metric_label = "Adaptive Accuracy" if metric == 'adaptive_accuracy' else "Transfer Accuracy"
    plt.ylabel(metric_label, fontsize=16)
    
    if task_name:
        title = f'{metric_label} by {ablation_label} - {task_name}'
        if eval_model != 'all':
            title += f' ({eval_model})'
        plt.title(title, fontsize=18)
    
    # Increase fontsize of labels and ticks
    plt.xticks(fontsize=14, rotation=45, ha='right')
    plt.yticks(fontsize=14)
    
    # Add value labels on top of bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', fontsize=12)
    
    plt.ylim(0, 1)
    plt.legend(title=ablation_label, title_fontsize=14, fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, format='png', bbox_inches='tight', dpi=300)
    
    plt.show()

def plot_multi_ablation_grid(
    df: pd.DataFrame,
    eval_model: str,
    metric: str = 'adaptive_accuracy',
    figsize: Tuple[int, int] = (18, 12),
    task_name: Optional[str] = None,
    save_path: Optional[str] = None
) -> None:
    """
    Plot a grid of bar charts showing multiple ablations side by side.
    
    Args:
        df: DataFrame with experiment results
        eval_model: Model to filter for (or 'all' for all models)
        metric: Metric to plot ('adaptive_accuracy' or 'transfer_accuracy')
        figsize: Figure size (width, height)
        task_name: Optional task name for title
        save_path: Optional path to save the figure
    """
    df_plot = clean_model_names(df)
    
    # Filter for the specified evaluation model
    if eval_model != 'all':
        df_plot = df_plot[df_plot['eval_model_name'].str.contains(eval_model)]
    
    # Create ablation columns with friendly names
    df_plot['Target CoT'] = df_plot['use_cot_target'].map({True: 'Yes', False: 'No'})
    df_plot['Generator CoT'] = df_plot['use_cot_in_context_attacker'].map({True: 'Yes', False: 'No'})
    df_plot['With Examples'] = df_plot['use_example'].map({True: 'Yes', False: 'No'})
    
    # Set up the grid
    ablation_columns = ['Target CoT', 'Generator CoT', 'With Examples']
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)
    
    # Set style
    sns.set_style("white")
    
    # Plot each ablation
    for i, ablation in enumerate(ablation_columns):
        ax = axes[i]
        sns.barplot(
            data=df_plot,
            x='generator_model_name',
            y=metric,
            hue=ablation,
            palette=['#1f77b4', '#ff7f0e'],
            ax=ax
        )
        
        # Adjust labels
        ax.set_xlabel('Generator Model', fontsize=16)
        if i == 0:
            metric_label = "Adaptive Accuracy" if metric == 'adaptive_accuracy' else "Transfer Accuracy"
            ax.set_ylabel(metric_label, fontsize=16)
        else:
            ax.set_ylabel('')
        
        ax.set_title(f'Effect of {ablation}', fontsize=18)
        
        # Adjust ticks
        ax.tick_params(axis='x', labelsize=14, rotation=45, labelrotation=45, ha='right')
        ax.tick_params(axis='y', labelsize=14)
        
        # Add value labels
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', fontsize=10)
        
        ax.set_ylim(0, 1)
        ax.legend(title=ablation, title_fontsize=14, fontsize=12)
    
    # Add overall title if task name is provided
    if task_name:
        title = f'Ablation Studies - {task_name}'
        if eval_model != 'all':
            title += f' ({eval_model})'
        fig.suptitle(title, fontsize=22)
        fig.subplots_adjust(top=0.9)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, format='png', bbox_inches='tight', dpi=300)
    
    plt.show()

def plot_negative_samples_comparison(
    df: pd.DataFrame,
    eval_model: str,
    generator_model: Optional[str] = None,
    metric: str = 'adaptive_accuracy',
    figsize: Tuple[int, int] = (10, 6),
    task_name: Optional[str] = None,
    save_path: Optional[str] = None
) -> None:
    """
    Plot the effect of different numbers of negative samples.
    
    Args:
        df: DataFrame with experiment results
        eval_model: Model to filter for
        generator_model: Optional generator model to filter for
        metric: Metric to plot ('adaptive_accuracy' or 'transfer_accuracy')
        figsize: Figure size (width, height)
        task_name: Optional task name for title
        save_path: Optional path to save the figure
    """
    df_plot = clean_model_names(df)
    
    # Filter for the specified models
    df_plot = df_plot[df_plot['eval_model_name'].str.contains(eval_model)]
    if generator_model:
        df_plot = df_plot[df_plot['generator_model_name'].str.contains(generator_model)]
    
    # Create configuration label
    df_plot['Configuration'] = (
        'CoT=' + df_plot['use_cot_target'].map({True: 'Yes', False: 'No'}) + 
        ', Examples=' + df_plot['use_example'].map({True: 'Yes', False: 'No'}) +
        ', Gen CoT=' + df_plot['use_cot_in_context_attacker'].map({True: 'Yes', False: 'No'})
    )
    
    # Set style
    sns.set_style("white")
    plt.figure(figsize=figsize, dpi=300)
    
    # Create the line plot
    sns.lineplot(
        data=df_plot,
        x='n_negative_samples',
        y=metric,
        hue='Configuration',
        style='generator_model_name',
        markers=True,
        dashes=False,
        linewidth=2,
        markersize=10
    )
    
    # Adjust labels and title
    plt.xlabel('Number of Negative Samples', fontsize=16)
    
    metric_label = "Adaptive Accuracy" if metric == 'adaptive_accuracy' else "Transfer Accuracy"
    plt.ylabel(metric_label, fontsize=16)
    
    if task_name:
        title = f'Effect of Negative Samples - {task_name}'
        if eval_model:
            title += f' ({eval_model})'
        plt.title(title, fontsize=18)
    
    # Increase fontsize of labels and ticks
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    
    # Use log scale for x-axis if there's a wide range of values
    if df_plot['n_negative_samples'].max() / df_plot['n_negative_samples'].min() > 4:
        plt.xscale('log', base=2)
        plt.xticks([2**i for i in range(6)], [f'$2^{i}$' for i in range(6)])
    
    plt.ylim(0, 1)
    plt.legend(title='Configuration', title_fontsize=14, fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, format='png', bbox_inches='tight', dpi=300)
    
    plt.show()


In [3]:

# Load experiment results
results_path = "/Users/davisbrown/adaptive_evals/cache/initial_eval_with_examples_cot_attacker_num_epochs_10.csv"
df = load_experiment_results(results_path)

# 1. Plot transfer matrix
plot_transfer_matrix(
    df, 
    metric='transfer_error_rate',
    task_name='LegalBench',
    save_path='figures/legal_transfer_matrix.pdf'
)

# 2. Plot ablation for a specific model
plot_ablation_bars(
    df,
    eval_model='GPT-4o',
    ablation_column='use_cot_target',
    task_name='LegalBench',
    save_path='figures/legal_cot_ablation.png'
)

# 3. Plot all ablations in a grid
plot_multi_ablation_grid(
    df,
    eval_model='GPT-4o',
    task_name='LegalBench',
    save_path='figures/legal_ablation_grid.png'
)

# 4. Plot effect of negative samples
plot_negative_samples_comparison(
    df,
    eval_model='GPT-4o',
    generator_model='GPT-4o',
    task_name='LegalBench',
    save_path='figures/legal_negative_samples.png'
)


KeyError: 'eval_model_name'

In [4]:
%debug

> /Users/davisbrown/opt/anaconda3/envs/inspect4/lib/python3.10/site-packages/pandas/core/indexes/base.py(3812)get_loc()
   3810             ):
   3811                 raise InvalidIndexError(key)
-> 3812             raise KeyError(key) from err
   3813         except TypeError:
   3814             # If we have a listlike key, _check_indexing_error will raise

*** NameError: name 'df' is not defined
*** NameError: name 'df' is not defined
*** NameError: name 'df' is not defined
*** NameError: name 'df' is not defined
*** NameError: name 'df' is not defined
*** NameError: name 'df' is not defined
